# Spike 04 — PageIndex Reasoning Graph

**Goal:** Feed the OCR'd Markdown into PageIndex and build a reasoning graph (tree structure) over the document.

**Time box:** 1-2 hours

**Question to answer:** Does PageIndex successfully index the bilingual OCR output and return a usable reasoning structure?

**Done when:** Documents are indexed in PageIndex and we can query them without errors.

---

### What is Vector-Less Reasoning RAG?

Traditional RAG:
```
Document → chunk into pieces → embed as vectors → store in vector DB
Query    → embed query       → find nearest chunks → send to LLM
```
Problem: chunking breaks tables and cross-references. Arabic embeddings are weaker than English ones.

PageIndex approach:
```
Document → build reasoning graph (understands structure)
Query    → reasoning traversal over graph → find relevant pages
```
This is why it suits our bilingual, table-heavy urban reports.

### API Key Setup
1. Go to [pageindex.ai](https://pageindex.ai)
2. Click "Try Now" or "Book a Demo"
3. Mention you're in a hackathon — many services give free credits for this
4. Add to `.env`: `PAGEINDEX_API_KEY=your_key_here`

---
### ⚠️ Important
PageIndex's API details aren't fully public. This notebook uses the most likely API shape  
based on their documentation. **Adjust endpoint URLs and payload keys** based on what  
their team sends you after signup.

## Step 1 — Imports & Setup

In [12]:
import requests
import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY not found in .env — get it from pageindex.ai")

# Confirmed working: https://api.pageindex.ai/health returns {"status":"ok"}
# No /v1 prefix on this API
BASE_URL = "https://api.pageindex.ai"

HEADERS = {
    "Authorization": f"Bearer {PAGEINDEX_API_KEY}",
    "Content-Type": "application/json"
}

print("✅ PageIndex client configured")
print(f"   Base URL: {BASE_URL}")
print(f"   API key loaded: {'Yes ✅' if PAGEINDEX_API_KEY else 'No ❌'}")

✅ PageIndex client configured
   Base URL: https://api.pageindex.ai
   API key loaded: Yes ✅


## Step 2 — Verify Connection & Discover API Schema

In [ ]:
# Step 2A — Confirm health endpoint (already proven to work)
r = requests.get(f"{BASE_URL}/health", timeout=10)
print(f"Health check: {r.status_code} — {r.text}")
assert r.status_code == 200, "Health check failed — check BASE_URL"
print("✅ Connected to PageIndex API")

## Step 2B — Discover the Real API Endpoints

PageIndex's endpoints aren't publicly documented. We probe common patterns to find what actually works before writing upload/query code.

In [13]:
# Probe common endpoint patterns — GET requests only (safe, no side effects)
# Run this to find what routes exist before attempting uploads
CANDIDATE_ENDPOINTS = [
    "/",
    "/docs",
    "/openapi.json",       # FastAPI auto-generated schema — goldmine if it exists
    "/redoc",
    "/api",
    "/api/v1",
    "/v1",
    "/documents",
    "/pages",
    "/index",
    "/collections",
    "/ingest",
    "/query",
    "/search",
]

print(f"{'Endpoint':<25} {'Status':>8}  Response preview")
print("-" * 70)

for path in CANDIDATE_ENDPOINTS:
    try:
        r = requests.get(f"{BASE_URL}{path}", headers=HEADERS, timeout=8)
        preview = r.text[:80].replace("\n", " ")
        marker = "✅" if r.status_code == 200 else ("⚠️ " if r.status_code in (401, 403, 405, 422) else "❌")
        print(f"{marker} {path:<23} {r.status_code:>8}  {preview}")
    except Exception as e:
        print(f"❌ {path:<23} {'ERR':>8}  {str(e)[:60]}")

print()
print("Key:")
print("  ✅ 200 = endpoint exists and responded")
print("  ⚠️  401/403 = exists but needs auth (try with HEADERS)")
print("  ⚠️  405 = exists but wrong HTTP method (try POST)")
print("  ❌ 404 = doesn't exist")

Endpoint                    Status  Response preview
----------------------------------------------------------------------
❌ /                            404  {"detail":"Not Found"}
✅ /docs                        200  {"documents":[],"total":0,"limit":50,"offset":0}
✅ /openapi.json                200  {"openapi":"3.1.0","info":{"title":"FastAPI","version":"0.1.0"},"paths":{"/tree"
✅ /redoc                       200       <!DOCTYPE html>     <html>     <head>     <title>FastAPI - ReDoc</title>   
❌ /api                         404  {"detail":"Not Found"}
❌ /api/v1                      404  {"detail":"Not Found"}
❌ /v1                          404  {"detail":"Not Found"}
❌ /documents                   404  {"detail":"Not Found"}
❌ /pages                       404  {"detail":"Not Found"}
❌ /index                       404  {"detail":"Not Found"}
❌ /collections                 404  {"detail":"Not Found"}
❌ /ingest                      404  {"detail":"Not Found"}
❌ /query                  

In [14]:
# Fetch full OpenAPI schema and print request body details for our 3 key endpoints
schema = requests.get(f"{BASE_URL}/openapi.json", headers=HEADERS, timeout=10).json()

TARGET_ENDPOINTS = ["/markdown", "/retrieval", "/chat/completions", "/doc", "/docs"]

for endpoint in TARGET_ENDPOINTS:
    if endpoint not in schema.get("paths", {}):
        continue
    print(f"\n{'='*60}")
    print(f"ENDPOINT: {endpoint}")
    for method, details in schema["paths"][endpoint].items():
        print(f"  Method : {method.upper()}")
        print(f"  Summary: {details.get('summary', '')}")

        # Request body
        if "requestBody" in details:
            body = details["requestBody"]
            content = body.get("content", {})
            for content_type, content_schema in content.items():
                print(f"  Body ({content_type}):")
                print(json.dumps(content_schema.get("schema", {}), indent=4))

        # Response shape
        responses = details.get("responses", {})
        for code, resp in responses.items():
            resp_content = resp.get("content", {})
            for ct, rs in resp_content.items():
                print(f"  Response {code} ({ct}):")
                print(json.dumps(rs.get("schema", {}), indent=4))


ENDPOINT: /markdown
  Method : POST
  Summary: Markdown To Tree
  Body (multipart/form-data):
{
    "$ref": "#/components/schemas/Body_markdown_to_tree_markdown_post"
}
  Response 200 (application/json):
{}
  Response 422 (application/json):
{
    "$ref": "#/components/schemas/HTTPValidationError"
}

ENDPOINT: /retrieval
  Method : POST
  Summary: Submit Pageindex Retrieval
  Body (application/json):
{
    "$ref": "#/components/schemas/PageIndexRetrievalRequest"
}
  Response 200 (application/json):
{}
  Response 422 (application/json):
{
    "$ref": "#/components/schemas/HTTPValidationError"
}

ENDPOINT: /chat/completions
  Method : POST
  Summary: Chat Completions
  Body (application/json):
{
    "$ref": "#/components/schemas/ChatCompletionRequest"
}
  Response 200 (application/json):
{}
  Response 422 (application/json):
{
    "$ref": "#/components/schemas/HTTPValidationError"
}

ENDPOINT: /doc
  Method : POST
  Summary: Page Index
  Body (multipart/form-data):
{
    "$ref": "#/comp

In [17]:
# The schema above uses $ref — resolve the actual field names for the components we need
components = schema.get("components", {}).get("schemas", {})

for component_name in ["PageIndexRetrievalRequest", "ChatCompletionRequest",
                        "Body_markdown_to_tree_markdown_post", "Body_page_index_doc_post"]:
    if component_name not in components:
        print(f"⚠️  {component_name} not found in schema")
        continue
    print(f"\n{'='*60}")
    print(f"SCHEMA: {component_name}")
    comp = components[component_name]
    props = comp.get("properties", {})
    required = comp.get("required", [])
    for field, details in props.items():
        req_marker = "* required" if field in required else "  optional"
        field_type = details.get("type") or details.get("$ref", "").split("/")[-1]
        desc = details.get("description", "")
        print(f"  {req_marker}  {field:<25} {field_type:<15} {desc}")


SCHEMA: PageIndexRetrievalRequest
  * required  doc_id                    string          
  * required  query                     string          
    optional  thinking                                  
    optional  teamspace_id                              

SCHEMA: ChatCompletionRequest
  * required  messages                  array           
    optional  stream                                    
    optional  temperature                               
    optional  doc_id                                    
    optional  enable_citations                          

SCHEMA: Body_markdown_to_tree_markdown_post
  * required  file                      string          
    optional  model                     string          
    optional  if_thinning               string          
    optional  thinning_threshold        integer         
    optional  summary_token_threshold   integer         
    optional  if_add_node_id            string          
    optional  if_add_node_summary 

## Step 3 — Load OCR Output

Read in the Markdown files produced by Spike 02 and 03.

In [15]:
AR_DIR = Path("../ocr_output/arabic")
EN_DIR = Path("../ocr_output/english")

ar_files = sorted(AR_DIR.glob("*.md"))
en_files = sorted(EN_DIR.glob("*.md"))

print(f"Arabic pages  : {len(ar_files)}")
print(f"English pages : {len(en_files)}")

if not ar_files and not en_files:
    print("\n❌ No OCR output found")
    print("   Run Spike 02 for Arabic and Spike 03 for English first")
else:
    # Preview one file
    sample = (ar_files or en_files)[0]
    content = sample.read_text(encoding="utf-8")
    print(f"\nSample ({sample.name}): {len(content)} chars")
    print(content[:300])

Arabic pages  : 14
English pages : 14

Sample (page_007.md): 1485 chars
# الكلمة الافتتاحية

تسعى هيئة تطوير منطقة المدينة المنورة إلى تحسين قابلية العيش في المدينة المنورة وتعزيز جودة حياة سكانها وأن تكون ضمن أفضل مدن العالم القابلة للعيش، وذلك تماشياً مع أهداف برنامج جودة الحياة، أحد برامج رؤية المملكة 2030. ومن هذا المنطلق يسعدني أن أقدم لكم هذا التقرير المبني على بي


## Step 4 — Upload a Single Markdown Document (Test)

We use `POST /markdown` — "Markdown To Tree".  
This is the perfect endpoint for us: our OCR output is already Markdown, so it goes in directly with no conversion.

Run the schema cell above first to confirm the exact payload field names.

In [18]:
def upload_markdown(content: str, name: str, metadata: dict = None) -> dict:
    """
    Upload a Markdown document via POST /markdown as multipart form data.
    Confirmed from 422 error: {"loc":["body","file"],"msg":"Field required"}
    → endpoint expects a 'file' field, NOT a JSON body.

    requests sets Content-Type: multipart/form-data automatically when
    using files= — do NOT pass Content-Type: application/json here.
    """
    file_bytes  = content.encode("utf-8")
    auth_headers = {"Authorization": HEADERS["Authorization"]}  # no Content-Type

    r = requests.post(
        f"{BASE_URL}/markdown",
        headers = auth_headers,
        files   = {"file": (f"{name}.md", file_bytes, "text/markdown")},
        timeout = 60
    )

    print(f"  Status: {r.status_code}")
    if r.status_code not in (200, 201, 202):
        print(f"  ❌ Error: {r.text[:500]}")
        return {}

    return r.json()


# ── Load OCR files ───────────────────────────────────────────────
AR_DIR = Path("../ocr_output/arabic")
EN_DIR = Path("../ocr_output/english")
ar_files = sorted(AR_DIR.glob("*.md"))
en_files = sorted(EN_DIR.glob("*.md"))

print(f"Arabic pages  : {len(ar_files)}")
print(f"English pages : {len(en_files)}")

# ── Test with ONE file first ─────────────────────────────────────
if en_files:
    test_file    = en_files[0]
    test_content = test_file.read_text(encoding="utf-8")

    print(f"\nUploading test document: {test_file.name}")
    result = upload_markdown(
        content = test_content,
        name    = f"Tranquil City EN — {test_file.stem}",
    )
    print("\nPageIndex response:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

Arabic pages  : 14
English pages : 14

Uploading test document: page_007.md
  Status: 200

PageIndex response:
{
  "success": true,
  "doc_name": "temp_Tranquil City EN — page_007",
  "doc_description": null,
  "structure": [
    {
      "title": "Opening Statement",
      "node_id": "0000",
      "text": "# Opening Statement\n\nThe Madinah Development Authority is committed to enhancing the livability of Madinah and improving the quality of life for its residents, aiming to position the city among the best livable cities in the world. This aligns with the goals of the Quality of Life Program, one of the initiatives under Saudi Vision 2030.\n\nIn this regard, I am pleased to present this report, which is based on reliable data and global standards concerning livability in the city of Madinah. This report reflects the efforts made by our development partners at both the national and local levels to achieve an ambitious vision aimed at improving living standards and enriching the lifesty

## Step 5 — Upload All Documents

Upload all Arabic + English OCR pages. Each page becomes its own reasoning tree node in PageIndex.

In [ ]:
# ── Diagnostic: see the raw response from one upload ────────────
print("=== RAW UPLOAD RESPONSE (one file) ===")
test_content = en_files[0].read_text(encoding="utf-8")
raw = requests.post(
    f"{BASE_URL}/markdown",
    headers = {"Authorization": HEADERS["Authorization"]},
    files   = {"file": ("diag_test.md", test_content.encode("utf-8"), "text/markdown")},
    timeout = 60
)
print(f"Status  : {raw.status_code}")
print(f"Response: {raw.text[:1000]}")   # full response — no truncation of keys

# ── Diagnostic: check if trees have their own listing endpoint ───
print("\n=== CHECKING /trees LISTING ===")
for path in ["/trees", "/tree", "/tree/list"]:
    r = requests.get(f"{BASE_URL}{path}", headers=HEADERS, timeout=8)
    print(f"  GET {path}: {r.status_code} — {r.text[:150]}")

In [ ]:
import time
import json as _json
from pathlib import Path as _Path

uploaded = []   # {"doc_name", "lang", "page", "node_count", "structure"}
failed   = []

all_docs = [
    *[(f, "en", "Madinah Tranquil City 2024 — English") for f in en_files],
    *[(f, "ar", "Madinah Tranquil City 2024 — Arabic")  for f in ar_files],
]

UPLOAD_CACHE = _Path("../ocr_output/upload_index.json")

# Skip re-uploading if we already have a saved index
if UPLOAD_CACHE.exists():
    uploaded = _json.loads(UPLOAD_CACHE.read_text(encoding="utf-8"))
    print(f"⏭  Loaded {len(uploaded)} uploads from cache — skipping re-upload")
    print(f"   Delete {UPLOAD_CACHE} to force re-upload")
else:
    for i, (file_path, lang, report_name) in enumerate(all_docs):
        content = file_path.read_text(encoding="utf-8")
        # Filename becomes doc_name on PageIndex (without extension, may get prefix)
        filename = f"{report_name.replace(' ', '_')}_{file_path.stem}.md"

        print(f"[{i+1}/{len(all_docs)}] {file_path.name} ({lang})...", end=" ")

        r = requests.post(
            f"{BASE_URL}/markdown",
            headers = {"Authorization": HEADERS["Authorization"]},
            files   = {"file": (filename, content.encode("utf-8"), "text/markdown")},
            timeout = 60
        )

        if r.status_code in (200, 201, 202):
            result = r.json()
            doc_name   = result.get("doc_name", filename)
            structure  = result.get("structure", [])
            uploaded.append({
                "doc_name"  : doc_name,
                "lang"      : lang,
                "page"      : file_path.stem,
                "node_count": len(structure),
                "structure" : structure      # the reasoning tree nodes
            })
            print(f"✅  doc_name={doc_name}  nodes={len(structure)}")
        else:
            print(f"❌  {r.status_code} — {r.text[:200]}")
            failed.append(file_path.name)

        time.sleep(0.5)

    # Save index so we don't need to re-upload during this spike
    UPLOAD_CACHE.write_text(
        _json.dumps(uploaded, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"\n✅ Uploaded : {len(uploaded)}  |  ❌ Failed: {len(failed)}")
    print(f"   Index saved to {UPLOAD_CACHE}")

## Step 5B — Inspect the Reasoning Tree

The `structure` field returned by `/markdown` IS the reasoning graph.  
PageIndex parsed each Markdown page into a tree of titled nodes — this is what gets searched during retrieval.

In [ ]:
# Show the tree structure for the first English and first Arabic doc
for lang_filter in ["en", "ar"]:
    docs_for_lang = [u for u in uploaded if u["lang"] == lang_filter]
    if not docs_for_lang:
        continue

    sample = docs_for_lang[0]
    print(f"\n{'='*60}")
    print(f"doc_name : {sample['doc_name']}")
    print(f"language : {sample['lang']}")
    print(f"page     : {sample['page']}")
    print(f"nodes    : {sample['node_count']}")
    print(f"\nReasoning Tree Nodes:")
    print("-" * 60)

    for node in sample.get("structure", []):
        node_id = node.get("node_id", "?")
        title   = node.get("title", "(no title)")
        text    = node.get("text", "")
        preview = text[:150].replace("\n", " ")
        print(f"  [{node_id}] {title}")
        print(f"         {preview}...")
        print()

# Summary across all uploaded docs
print(f"\n{'='*60}")
print("TREE STRUCTURE SUMMARY")
print(f"{'='*60}")
total_nodes = sum(u["node_count"] for u in uploaded)
en_docs = [u for u in uploaded if u["lang"] == "en"]
ar_docs = [u for u in uploaded if u["lang"] == "ar"]
print(f"English pages : {len(en_docs)}  |  Total nodes: {sum(u['node_count'] for u in en_docs)}")
print(f"Arabic pages  : {len(ar_docs)}  |  Total nodes: {sum(u['node_count'] for u in ar_docs)}")
print(f"Grand total   : {len(uploaded)} pages  |  {total_nodes} reasoning tree nodes")

## Step 6 — Submit a Retrieval Query

`POST /retrieval` is async: it returns a `retrieval_id`, then we poll `GET /retrieval/{retrieval_id}` for results.

We also have `POST /chat/completions` which is OpenAI-compatible — PageIndex handles retrieval + answer generation in one call. We'll test both.

In [ ]:
import time as _time

# ── Approach A: POST /retrieval (async, returns raw passages) ────
def retrieve(query: str, doc_ids: list = None, top_k: int = 5) -> dict:
    """
    Submit a retrieval query. PageIndex processes it asynchronously.
    Returns retrieval_id — use get_retrieval_result() to fetch results.
    Adjust payload keys based on what the schema cell showed.
    """
    payload = {"query": query, "top_k": top_k}
    if doc_ids:
        payload["doc_ids"] = doc_ids   # narrow to specific docs if needed

    r = requests.post(f"{BASE_URL}/retrieval", headers=HEADERS, json=payload, timeout=30)
    print(f"  Status: {r.status_code}")
    if r.status_code not in (200, 201, 202):
        print(f"  ❌ {r.text[:400]}")
        return {}
    return r.json()


def get_retrieval_result(retrieval_id: str, max_wait: int = 30) -> dict:
    """Poll until retrieval is complete or timeout."""
    for attempt in range(max_wait):
        r = requests.get(f"{BASE_URL}/retrieval/{retrieval_id}", headers=HEADERS, timeout=10)
        data = r.json()
        status = data.get("status", "unknown")
        if status in ("completed", "done", "success") or "results" in data or "passages" in data:
            return data
        print(f"  [{attempt+1}] status={status} — waiting...")
        _time.sleep(1)
    return {"error": "timeout"}


# ── Test retrieval ────────────────────────────────────────────────
print("=== APPROACH A: POST /retrieval ===")
query = "What are the livability indicators for Madinah in 2024?"
print(f"Query: {query}\n")

retrieval_response = retrieve(query)
print("Retrieval submission response:")
print(json.dumps(retrieval_response, indent=2, ensure_ascii=False))

# If async — poll for results
retrieval_id = (retrieval_response.get("retrieval_id")
                or retrieval_response.get("id")
                or retrieval_response.get("task_id"))

if retrieval_id:
    print(f"\nPolling for results (retrieval_id={retrieval_id})...")
    result = get_retrieval_result(retrieval_id)
    print("\nRetrieval result:")
    print(json.dumps(result, indent=2, ensure_ascii=False)[:2000])
else:
    # Some APIs return results immediately
    print("\n(Results returned immediately — no async polling needed)")

## Step 7 — Spike Summary

## Step 7 — Chat Completions (OpenAI-Compatible RAG in One Call)

PageIndex exposes `POST /chat/completions` in OpenAI format.  
This means the OpenAI SDK can point directly at PageIndex — retrieval + answer generation happens server-side.  
This is the cleanest integration path for Spike 05.

In [ ]:
from openai import OpenAI

# Point the OpenAI SDK at PageIndex instead of OpenAI
# Same pattern as Groq — just change base_url and api_key
pageindex_client = OpenAI(
    api_key  = PAGEINDEX_API_KEY,
    base_url = BASE_URL
)

# ── Test: English question ───────────────────────────────────────
print("=== APPROACH B: POST /chat/completions (OpenAI-compatible) ===\n")

question_en = "What are the livability indicators for Madinah in 2024?"
print(f"Q (EN): {question_en}")

response_en = pageindex_client.chat.completions.create(
    model    = "pageindex",       # adjust model name if schema shows something different
    messages = [
        {"role": "system", "content": "You are an expert on Madinah urban development reports. Answer using only the indexed documents. Cite your sources."},
        {"role": "user",   "content": question_en}
    ]
)
print(f"\nA: {response_en.choices[0].message.content}")

# ── Test: Arabic question ────────────────────────────────────────
print("\n" + "="*60)
question_ar = "ما هي مؤشرات قابلية العيش في المدينة المنورة؟"
print(f"Q (AR): {question_ar}")

response_ar = pageindex_client.chat.completions.create(
    model    = "pageindex",
    messages = [
        {"role": "system", "content": "أنت خبير في تقارير التنمية الحضرية للمدينة المنورة. أجب بالعربية فقط باستخدام الوثائق المفهرسة. اذكر المصادر."},
        {"role": "user",   "content": question_ar}
    ]
)
print(f"\nA: {response_ar.choices[0].message.content}")

In [ ]:
print("=" * 50)
print("SPIKE 04 — RESULTS")
print("=" * 50)
print(f"Total documents uploaded : {len(uploaded)}")
print(f"Arabic pages indexed     : {sum(1 for d in uploaded if '_ar_' in d)}")
print(f"English pages indexed    : {sum(1 for d in uploaded if '_en_' in d)}")
print(f"Failed uploads           : {len(failed)}")
print()

if len(uploaded) > 0 and len(failed) == 0:
    print("✅ SPIKE PASSED — Knowledge base ready for Spike 05 (retrieval)")
elif len(uploaded) > 0:
    print("⚠️  SPIKE PARTIAL — Some uploads failed, check error messages above")
else:
    print("❌ SPIKE FAILED — No documents uploaded")
    print("   Most likely cause: wrong BASE_URL or API schema")
    print("   Fix: Contact PageIndex support and get their exact API docs")